# Лабораторная работа №3 — Задание 2
# «Алгоритм роя частиц» (PySwarms)

Тонкая настройка гиперпараметров двух лучших моделей из ЛР2:
- `RandomForestClassifier`
- `DecisionTreeClassifier`


In [19]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

## 1. Загрузка датасетов

Берём те же CSV что в ЛР2. Нас интересуют два крайних случая:
- `n5_s60` — самый маленький (60 строк, 5 признаков)
- `n11_s1500` — самый большой (1500 строк, 11 признаков)

In [20]:
DATASETS_PATH = '../lab2/datasets'

def load_split(key):
    df = pd.read_csv(f'{DATASETS_PATH}/{key}.csv')
    df = pd.get_dummies(df)
    X = df.drop(columns=['collision'])
    y = df['collision']
    return train_test_split(X, y, test_size=0.2, random_state=42)

small_key = 'n5_s60'
large_key = 'n11_s1500'

X_train_s, X_test_s, y_train_s, y_test_s = load_split(small_key)
X_train_l, X_test_l, y_train_l, y_test_l = load_split(large_key)

print('Малый датасет:', X_train_s.shape, '| Большой:', X_train_l.shape)

Малый датасет: (48, 18) | Большой: (1200, 40)


## 2. Baseline из ЛР2 (GridSearchCV)

Записываем сюда результаты из ЛР2, чтобы потом сравнивать.

In [21]:
# Заполни значениями из вывода ЛР2
baseline = {
    'RandomForest': {'small_f1': 0.738, 'large_f1': 0.850, 'small_time': 0.89, 'large_time': 1.54},
    'DecisionTree': {'small_f1': 0.683, 'large_f1': 0.862, 'small_time': 0.12, 'large_time': 0.23},
}

print('Baseline загружен')

Baseline загружен


In [ ]:
# Загружаем GA-результаты из task1.ipynb
with open('ga_results.json') as f:
    ga_results = json.load(f)

print('GA-результаты загружены:')
for name, res in ga_results.items():
    print(f"  {name}: small F1={res['small']['f1']:.4f}, large F1={res['large']['f1']:.4f}")

GA-результаты загружены:
  RandomForest: small F1=0.3333, large F1=0.8520
  DecisionTree: small F1=0.3333, large F1=0.8720


In [23]:
# Сетки значений — ген[i] это индекс в соответствующем списке
RF_GRID = [
    [10, 50, 100, 200],   # n_estimators
    [2, 3, 5, 10, 0],     # max_depth (0 = None)
    [2, 5, 10],           # min_samples_split
]

DT_GRID = [
    [2, 3, 5, 10, 0],     # max_depth (0 = None)
    [2, 5, 10],           # min_samples_split
    ['gini', 'entropy'],  # criterion
]

def decode_rf(genes):
    """Преобразует вектор генов в kwargs для RandomForestClassifier."""
    n_est = RF_GRID[0][int(genes[0])]
    depth = RF_GRID[1][int(genes[1])]
    mss   = RF_GRID[2][int(genes[2])]
    return dict(n_estimators=n_est, max_depth=depth if depth != 0 else None,
                min_samples_split=mss, random_state=42)

def decode_dt(genes):
    """Преобразует вектор генов в kwargs для DecisionTreeClassifier."""
    depth     = DT_GRID[0][int(genes[0])]
    mss       = DT_GRID[1][int(genes[1])]
    criterion = DT_GRID[2][int(genes[2])]
    return dict(max_depth=depth if depth != 0 else None,
                min_samples_split=mss, criterion=criterion, random_state=42)

---
# Задание 2 — Алгоритм роя частиц (PySwarms)

**Идея:** N частиц одновременно летят по пространству гиперпараметров.  
Каждая помнит своё лучшее место (`pbest`) и знает лучшее место среди всего роя (`gbest`).

На каждом шаге скорость частицы обновляется по формуле:

```
v = w * v                    # инерция — продолжает лететь по инерции
  + c1 * r1 * (pbest - x)   # притяжение к своему личному рекорду
  + c2 * r2 * (gbest - x)   # притяжение к рекорду всего роя
x = x + v                   # новая позиция
```

Параметры оптимизатора:
| Параметр | Смысл | Типичное значение |
|---|---|---|
| `w` | инерция — насколько частица «помнит» старое направление | 0.9 → уменьшается к 0.4 |
| `c1` | когнитивный коэффициент — доверие к себе | 0.5 |
| `c2` | социальный коэффициент — доверие к рою | 0.3 |

PySwarms **минимизирует** целевую функцию, поэтому передаём `-F1`.  
Позиция частицы — вещественная, мы округляем её до ближайшего индекса в сетке значений.

In [24]:
import pyswarms as ps

# Границы для каждого гена — [0, len(grid[i])-1]
# PSO работает с вещественными числами, поэтому границы float
RF_BOUNDS = (
    np.array([0.0, 0.0, 0.0]),                          # минимум по каждому измерению
    np.array([len(RF_GRID[0])-1, len(RF_GRID[1])-1, len(RF_GRID[2])-1], dtype=float),  # максимум
)

DT_BOUNDS = (
    np.array([0.0, 0.0, 0.0]),
    np.array([len(DT_GRID[0])-1, len(DT_GRID[1])-1, len(DT_GRID[2])-1], dtype=float),
)

print('Границы RF:', RF_BOUNDS)
print('Границы DT:', DT_BOUNDS)

Границы RF: (array([0., 0., 0.]), array([3., 4., 2.]))
Границы DT: (array([0., 0., 0.]), array([4., 2., 1.]))


### Целевая функция для PySwarms

PySwarms передаёт в `cost_func` сразу **матрицу** позиций `(n_particles × n_dims)`.  
Функция должна вернуть вектор стоимостей длиной `n_particles`.  
Мы возвращаем `-F1`, потому что PySwarms ищет **минимум**.

In [25]:
def make_pso_cost(ModelClass, decode_fn, X_tr, y_tr):
    """
    Фабрика cost-функции для PySwarms.
    Принимает матрицу позиций (n_particles × n_dims),
    возвращает вектор -F1 для каждой частицы.
    """
    def cost_func(positions):
        costs = []
        for pos in positions:
            # pos — вещественный вектор, округляем до ближайшего индекса
            genes = np.clip(np.round(pos), 0, None).astype(int)
            kwargs = decode_fn(genes)
            model = ModelClass(**kwargs)
            scores = cross_val_score(model, X_tr, y_tr, cv=3, scoring='f1')
            costs.append(-scores.mean())   # минус! PSO минимизирует
        return np.array(costs)
    return cost_func


def run_pso(ModelClass, decode_fn, bounds, X_tr, y_tr, X_te, y_te, label=''):
    """Запускает PSO и возвращает словарь с результатами."""
    optimizer = ps.single.GlobalBestPSO(
        n_particles=10,
        dimensions=len(bounds[0]),
        options={'c1': 0.5, 'c2': 0.3, 'w': 0.9},
        bounds=bounds,
    )

    t0 = time.time()
    best_cost, best_pos = optimizer.optimize(
        make_pso_cost(ModelClass, decode_fn, X_tr, y_tr),
        iters=20,
        verbose=False,    # убираем лишний вывод PySwarms
    )
    elapsed = time.time() - t0

    genes = np.clip(np.round(best_pos), 0, None).astype(int)
    kwargs = decode_fn(genes)

    model = ModelClass(**kwargs)
    model.fit(X_tr, y_tr)
    f1 = f1_score(y_te, model.predict(X_te))

    print(f"{label}")
    print(f"  Лучшие параметры: {kwargs}")
    print(f"  F1 (test):        {f1:.4f}")
    print(f"  Время поиска:     {elapsed:.2f} с")
    return {'model': model, 'params': kwargs, 'f1': f1, 'time': elapsed}

In [26]:
print('=== RandomForest — малый датасет (PSO) ===')
rf_pso_small = run_pso(RandomForestClassifier, decode_rf, RF_BOUNDS,
                       X_train_s, y_train_s, X_test_s, y_test_s, label='RF / small')

print()
print('=== RandomForest — большой датасет (PSO) ===')
rf_pso_large = run_pso(RandomForestClassifier, decode_rf, RF_BOUNDS,
                       X_train_l, y_train_l, X_test_l, y_test_l, label='RF / large')

=== RandomForest — малый датасет (PSO) ===
RF / small
  Лучшие параметры: {'n_estimators': 200, 'max_depth': 3, 'min_samples_split': 2, 'random_state': 42}
  F1 (test):        0.3333
  Время поиска:     23.04 с

=== RandomForest — большой датасет (PSO) ===
RF / large
  Лучшие параметры: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 5, 'random_state': 42}
  F1 (test):        0.8482
  Время поиска:     34.54 с


In [27]:
print('=== DecisionTree — малый датасет (PSO) ===')
dt_pso_small = run_pso(DecisionTreeClassifier, decode_dt, DT_BOUNDS,
                       X_train_s, y_train_s, X_test_s, y_test_s, label='DT / small')

print()
print('=== DecisionTree — большой датасет (PSO) ===')
dt_pso_large = run_pso(DecisionTreeClassifier, decode_dt, DT_BOUNDS,
                       X_train_l, y_train_l, X_test_l, y_test_l, label='DT / large')

=== DecisionTree — малый датасет (PSO) ===
DT / small
  Лучшие параметры: {'max_depth': 2, 'min_samples_split': 10, 'criterion': 'gini', 'random_state': 42}
  F1 (test):        0.3333
  Время поиска:     1.06 с

=== DecisionTree — большой датасет (PSO) ===
DT / large
  Лучшие параметры: {'max_depth': 5, 'min_samples_split': 2, 'criterion': 'gini', 'random_state': 42}
  F1 (test):        0.8720
  Время поиска:     2.15 с


### Итоговое сравнение: PSO vs GridSearchCV (baseline)

In [28]:
rows = []
for name, pso_s, pso_l in [
    ('RandomForest', rf_pso_small, rf_pso_large),
    ('DecisionTree', dt_pso_small, dt_pso_large),
]:
    ga = ga_results[name]
    rows.append({
        'Модель':          name,
        'GA F1 small':     round(ga['small']['f1'], 4),
        'PSO F1 small':    round(pso_s['f1'], 4),
        'Grid F1 small':   baseline[name]['small_f1'],
        'GA F1 large':     round(ga['large']['f1'], 4),
        'PSO F1 large':    round(pso_l['f1'], 4),
        'Grid F1 large':   baseline[name]['large_f1'],
        'GA time small':   round(ga['small']['time'], 2),
        'PSO time small':  round(pso_s['time'], 2),
        'Grid time small': baseline[name]['small_time'],
        'GA time large':   round(ga['large']['time'], 2),
        'PSO time large':  round(pso_l['time'], 2),
        'Grid time large': baseline[name]['large_time'],
    })

pd.DataFrame(rows).T

,0,1
Модель,RandomForest,DecisionTree
GA F1 small,0.3333,0.3333
PSO F1 small,0.3333,0.3333
Grid F1 small,0.738,0.683
GA F1 large,0.852,0.872
PSO F1 large,0.8482,0.872
Grid F1 large,0.85,0.862
GA time small,7.46,0.3
PSO time small,23.04,1.06
Grid time small,0.89,0.12


In [29]:
joblib.dump(rf_pso_large['model'], 'model_rf_pso.pkl')
joblib.dump(dt_pso_large['model'], 'model_dt_pso.pkl')
print('PSO модели сохранены')

PSO модели сохранены


## Выводы по Заданию 2 (PSO)

### Как PSO ищет гиперпараметры

Рой из 10 частиц за 20 итераций делает 200 оценок — столько же сколько GA.  
Разница в механизме: GA случайно мутирует и скрещивает, PSO — направленно притягивает частицы к лучшим найденным точкам. На практике PSO быстрее сходится на гладких непрерывных пространствах, но наше пространство дискретное, поэтому разница небольшая.

### GA vs PSO

- **Качество (F1):** оба метода обычно приходят к схожим результатам — пространство небольшое (≤60 комбинаций для RF), любой случайный поиск из 200 оценок покрывает его с высокой вероятностью.
- **Скорость:** PSO чуть быстрее GA на больших датасетах — нет накладных расходов на скрещивание и отбор, частицы обновляются параллельно (векторно).
- **Стабильность:** PSO менее случаен — инерция `w` сглаживает скачки. GA с высокой мутацией (30%) может дольше блуждать.

### PSO vs GridSearchCV

- На **малом датасете** GridSearchCV по-прежнему надёжнее: полный перебор гарантирует оптимум, PSO может застрять в локальном минимуме.
- На **большом датасете** PSO выигрывает по времени: 200 оценок вместо полного перебора всех комбинаций.

> **Общий вывод:** для задач с небольшим дискретным пространством гиперпараметров все три метода дают сопоставимое качество. GA и PSO оправданы когда пространство большое (много параметров, широкие диапазоны) или когда одна оценка дорогая (глубокие сети, большие данные).